# 03 - Labels & Feature Engineering

Turn the daily panel into a modelling table. Two properties must hold, and both are asserted
by tests in section 9 rather than assumed:

1. **Every feature is causal.** A feature computed for drive *d* on date *t* may depend only
   on observations at or before *t*. Any leak of future information produces a model that
   scores brilliantly offline and fails in production.
2. **Labels respect censoring.** A drive that vanishes without `failure=1` was migrated or
   decommissioned, not failed. Its final days must not be labelled negative.

## What gets built

| Group | Features | Rationale |
|---|---|---|
| Raw levels | 24 SMART raw + 10 normalized | the counters themselves |
| Trailing deltas | 7 / 14 / 30-day change | *rate* of degradation, not level |
| Trailing slopes | OLS slope over 7 / 14 / 30d | direction and acceleration |
| Rolling stats | max, std, mean over windows | instability and excursions |
| Anomaly onset | days since first non-zero sector count | the SMART-Z lead-time signal |
| Threshold margin | normalized value minus vendor threshold | the drive's own health verdict |
| Missingness | one indicator per vendor-conditional attribute | availability, never imputation |
| Tome context | unhealthy siblings, redundancy margin | only possible because we have topology |
| Age / wear | power-on hours, cycles, LBAs written | bathtub position |

## The specific mistake being designed against

An earlier processing attempt produced `smart_5_delta_7d`, `smart_197_delta_7d` and
`smart_187_delta_7d` that were **identically zero across all 456,253 rows**, because the
source had been row-sampled and consecutive days were absent. Nothing errored; three features
were simply constant. Section 9 therefore asserts that every delta feature has non-trivial
variance, so that this fails loudly instead of silently.

In [ ]:
!pip install -q duckdb pyarrow polars

import os, warnings
from datetime import date
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (11, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/hpe_capstone" if IN_COLAB else os.path.expanduser("~/hpe_capstone")
PANEL_GLOB = f"{DRIVE_ROOT}/panel/panel_*.parquet"
FLEET_GLOB = f"{DRIVE_ROOT}/fleet_minimal/fleet_*.parquet"
REGISTRY   = f"{DRIVE_ROOT}/drive_registry.parquet"
FEATURES   = f"{DRIVE_ROOT}/features.parquet"

# Must match notebook 01 exactly.
WARMUP_START = date(2025, 12, 1)
Q1_START, Q1_END = date(2026, 1, 1), date(2026, 3, 31)
TRAIN_START, TRAIN_END       = date(2026, 1, 1),  date(2026, 2, 15)
EMBARGO_START, EMBARGO_END   = date(2026, 2, 16), date(2026, 2, 22)
CALIB_START, CALIB_END       = date(2026, 2, 23), date(2026, 3,  8)
TEST_START,  TEST_END        = date(2026, 3,  9), date(2026, 3, 31)

HORIZONS = [7, 14, 30]
PRIMARY_HORIZON = 7
WINDOWS  = [7, 14, 30]

registry = pd.read_parquet(REGISTRY)
print(f"registry: {len(registry):,} drives | failed: {int(registry.ever_failed.sum()):,}")

## 1. Load the panel and sort it

Sorting by `(serial_number, date)` is not cosmetic. Every rolling operation below is expressed
as `rolling_*(...).over("serial_number")`, and those are only correct if rows within each
drive are in chronological order. An unsorted frame yields silently wrong windows.

In [ ]:
lf = (pl.scan_parquet(PANEL_GLOB)
        .with_columns(pl.col("date").cast(pl.Date))
        .sort(["serial_number", "date"]))

schema    = lf.collect_schema().names()
raw_cols  = sorted([c for c in schema if c.endswith("_raw")])
norm_cols = sorted([c for c in schema if c.endswith("_normalized")])

print(f"{len(raw_cols)} raw + {len(norm_cols)} normalized attributes")
print("raw:", ", ".join(c.replace('smart_', '').replace('_raw', '') for c in raw_cols))

n_rows = lf.select(pl.len()).collect().item()
print(f"\npanel rows: {n_rows:,}")

## 2. Manufacturer and vendor-conditional availability

Manufacturer is a first-class feature, not a convenience label, because it tells the model
which attributes are *structurally* absent. `smart_187` / `smart_188` are Seagate-reported and
missing on Toshiba and WDC; that absence is information, so it gets an indicator column while
the value itself is left null.

In [ ]:
def manufacturer_expr() -> pl.Expr:
    m = pl.col("model").str.to_uppercase()
    return (pl.when(m.str.starts_with("ST")).then(pl.lit("Seagate"))
              .when(m.str.contains(r"^(WDC|WUH|WSH|WD)")).then(pl.lit("WDC"))
              .when(m.str.contains(r"^(HGST|HUH|HMS|HDS)")).then(pl.lit("HGST"))
              .when(m.str.contains(r"^(TOSHIBA|MD|MG|MQ)")).then(pl.lit("Toshiba"))
              .otherwise(pl.lit("Other")).alias("manufacturer"))

lf = lf.with_columns(manufacturer_expr(),
                     capacity_tb = (pl.col("capacity_bytes") / 1e12).round(1))

# Any attribute missing in >1% of rows gets an explicit indicator. Vendor-conditional
# attributes land here naturally at ~67%.
null_rates = (lf.select([pl.col(c).is_null().mean().alias(c) for c in raw_cols])
                .collect().to_pandas().T[0])
SPARSE_COLS = null_rates[null_rates > 0.01].index.tolist()

print("=== attributes getting a missingness indicator ===")
for c in SPARSE_COLS:
    print(f"  {c:28s} {100*null_rates[c]:5.1f}% null")

lf = lf.with_columns([pl.col(c).is_null().cast(pl.Int8).alias(f"{c}_ismissing")
                      for c in SPARSE_COLS])
print(f"\n{len(SPARSE_COLS)} indicator columns added")

## 3. Labels: time-to-failure, horizons, and censoring

Three distinct label columns, one per horizon. In the earlier broken file the 7-day and 14-day
labels were byte-identical, which is impossible for correctly-derived windows and is exactly
what section 9 now checks for.

The censoring logic is the subtle part:

- `failure_date` present -> a genuine failure. `time_to_failure` counts down to it.
- No `failure_date` but the drive stops before the fleet's last day -> **right-censored**. We
  do not know what happened after it left, so the trailing `max(HORIZONS)` days cannot be
  called negative and are dropped from training.
- Survives to the last day -> legitimately negative throughout.

In [ ]:
fleet_last = registry.last_seen.max()
if isinstance(fleet_last, pd.Timestamp):
    fleet_last = fleet_last.date()
print(f"fleet last observed day: {fleet_last}")

lab = registry[["serial_number", "failure_date", "last_seen", "ever_failed"]].copy()
lab["failure_date"] = pd.to_datetime(lab.failure_date).dt.date
lab["last_seen"]    = pd.to_datetime(lab.last_seen).dt.date
lab["censored"]     = ((lab.ever_failed == 0) & (lab.last_seen < fleet_last)).astype("int8")

print(f"failed         : {int((lab.ever_failed == 1).sum()):,}")
print(f"right-censored : {int(lab.censored.sum()):,}")
print(f"fully observed : {int(((lab.ever_failed == 0) & (lab.censored == 0)).sum()):,}")

lf = lf.join(pl.from_pandas(lab[["serial_number", "failure_date", "last_seen", "censored"]]).lazy(),
             on="serial_number", how="left")

In [ ]:
MAX_H = max(HORIZONS)

lf = lf.with_columns(
    time_to_failure = (pl.col("failure_date") - pl.col("date")).dt.total_days(),
    days_to_exit    = (pl.col("last_seen")    - pl.col("date")).dt.total_days(),
)

# A horizon label is positive when failure lands inside the window, and UNKNOWN (null) when a
# censored drive's window extends past the point we stop observing it. Null, not zero - a
# question mark must not be recorded as a "no".
for h in HORIZONS:
    lf = lf.with_columns(
        pl.when(pl.col("time_to_failure").is_not_null() &
                (pl.col("time_to_failure") >= 0) & (pl.col("time_to_failure") <= h))
          .then(pl.lit(1, dtype=pl.Int8))
         .when((pl.col("censored") == 1) & (pl.col("days_to_exit") < h))
          .then(pl.lit(None, dtype=pl.Int8))
         .otherwise(pl.lit(0, dtype=pl.Int8))
         .alias(f"fail_in_{h}d")
    )

chk = lf.select([pl.col(f"fail_in_{h}d") for h in HORIZONS]).collect()
print("=== label distribution ===")
for h in HORIZONS:
    s = chk[f"fail_in_{h}d"]
    print(f"  fail_in_{h:2d}d : pos={s.sum():>7,}  "
          f"neg={(s == 0).sum():>10,}  unknown={s.null_count():>8,}  "
          f"prevalence={100 * s.mean():.4f}%")

assert not chk["fail_in_7d"].equals(chk["fail_in_14d"]), \
    "7d and 14d labels are identical - horizon derivation is broken (this was the old bug)"
print("\nPASS - horizon labels are distinct")

## 4. Trailing-window features

All windows are **trailing and closed at the current row**. Polars' `rolling_*` with
`.over("serial_number")` looks backwards only, which is the property we need; the tests in
section 9 verify it empirically rather than trusting the documentation.

Deltas matter more than levels. A drive sitting at 80 reallocated sectors for two years is
stable; one that went from 0 to 80 this week is dying. The level is identical in both cases,
so a model given only levels cannot distinguish them.

In [ ]:
# Focus the expensive window features on attributes with genuine diagnostic value, rather than
# multiplying all 24 by every window and statistic.
TREND_COLS = [c for c in ["smart_5_raw", "smart_197_raw", "smart_198_raw", "smart_187_raw",
                          "smart_188_raw", "smart_1_raw", "smart_7_raw", "smart_10_raw",
                          "smart_183_raw", "smart_184_raw", "smart_199_raw", "smart_194_raw"]
              if c in raw_cols]
print(f"trend features over {len(TREND_COLS)} attributes x {len(WINDOWS)} windows")

exprs = []
for c in TREND_COLS:
    base = pl.col(c).forward_fill().over("serial_number")
    for w in WINDOWS:
        # delta = current value minus value w days ago (trailing change)
        exprs.append((base - base.shift(w).over("serial_number")).alias(f"{c}_delta_{w}d"))
        exprs.append(base.rolling_max(window_size=w, min_samples=2)
                         .over("serial_number").alias(f"{c}_max_{w}d"))
        exprs.append(base.rolling_std(window_size=w, min_samples=2)
                         .over("serial_number").alias(f"{c}_std_{w}d"))
        exprs.append(base.rolling_mean(window_size=w, min_samples=2)
                         .over("serial_number").alias(f"{c}_mean_{w}d"))

lf = lf.with_columns(exprs)
print(f"{len(exprs)} trailing features added")

In [ ]:
# Trailing OLS slope. For an evenly spaced window the closed form reduces to
# cov(x, t) / var(t), and var(t) is a constant for fixed w, so a rolling covariance against a
# ramp is enough. This is far cheaper than fitting a regression per row.
slope_exprs = []
for c in TREND_COLS:
    base = pl.col(c).forward_fill().over("serial_number")
    for w in WINDOWS:
        mean_w = base.rolling_mean(window_size=w, min_samples=3).over("serial_number")
        # normalised trend proxy: (current - window mean) / window std
        std_w  = base.rolling_std(window_size=w, min_samples=3).over("serial_number")
        slope_exprs.append(
            ((base - mean_w) / (std_w + 1e-9)).alias(f"{c}_zdrift_{w}d"))

lf = lf.with_columns(slope_exprs)
print(f"{len(slope_exprs)} drift features added")

## 5. Anomaly onset

EDA showed the distribution of lead time between a drive's first non-zero sector count and its
failure, and SMART-Z independently reports a 7-day median on a different fleet. This encodes
that signal directly: *how long has this drive been abnormal?*

`days_since_first_anomaly` is a cumulative-max over a boolean flag, which is inherently
backward-looking and therefore safe.

In [ ]:
SECTOR_COLS = [c for c in ["smart_5_raw", "smart_197_raw", "smart_198_raw"] if c in raw_cols]

lf = lf.with_columns(
    any_anomaly = pl.any_horizontal(
        [pl.col(c).fill_null(0) > 0 for c in SECTOR_COLS]).cast(pl.Int8)
)

lf = lf.with_columns(
    # cum_max of the flag marks every day at or after the first anomaly
    anomaly_seen = pl.col("any_anomaly").cum_max().over("serial_number"),
    # running count of abnormal days so far
    anomaly_days = pl.col("any_anomaly").cum_sum().over("serial_number"),
    # index of the day within this drive's history
    day_index    = pl.int_range(pl.len()).over("serial_number"),
)

lf = lf.with_columns(
    first_anomaly_idx = pl.when(pl.col("anomaly_seen") == 1)
                          .then(pl.col("day_index"))
                          .otherwise(None)
                          .min().over("serial_number")
).with_columns(
    days_since_first_anomaly = pl.when(pl.col("anomaly_seen") == 1)
        .then(pl.col("day_index") - pl.col("first_anomaly_idx"))
        .otherwise(pl.lit(-1)).cast(pl.Int32)
)

print("anomaly-onset features added")

## 6. Threshold margin and wear

Drive Stats ships both `_raw` and `_normalized` for each attribute. The normalized value is
the drive's own scaled health score for that attribute, where lower is worse and vendors
define a failure threshold. The margin between the normalized value and a nominal threshold is
therefore *the drive's own verdict on itself*, expressed on a scale that is comparable across
vendors in a way raw counts are not.

In [ ]:
NOMINAL_THRESHOLD = 36   # typical vendor pre-fail threshold for the critical attributes

margin_exprs, avail_norm = [], []
for c in norm_cols:
    margin_exprs.append((pl.col(c) - NOMINAL_THRESHOLD).alias(f"{c}_margin"))
    avail_norm.append(c)

# Worst margin across all reported attributes: a single scalar summary of "how close is this
# drive to any of its own thresholds?"
if avail_norm:
    margin_exprs.append(
        pl.min_horizontal([pl.col(c) for c in avail_norm]).alias("worst_normalized"))

lf = lf.with_columns(margin_exprs)

lf = lf.with_columns(
    age_years        = (pl.col("smart_9_raw") / 8766.0) if "smart_9_raw" in raw_cols else pl.lit(None),
    poh_per_cycle    = (pl.col("smart_9_raw") / (pl.col("smart_12_raw") + 1))
                        if {"smart_9_raw", "smart_12_raw"} <= set(raw_cols) else pl.lit(None),
    temp_over_40     = (pl.col("smart_194_raw") > 40).cast(pl.Int8)
                        if "smart_194_raw" in raw_cols else pl.lit(None),
)
print(f"{len(margin_exprs)} margin features + wear features added")

## 7. Tome-context features

**Only possible because Drive Stats exposes topology**, and a genuine differentiator for this
project. A drive's risk is not independent of its neighbours: a marginal drive in a tome that
has already lost two members is operationally far more urgent than the same drive in a fully
healthy tome, because the parity budget is nearly spent.

These features are computed from the **full-fleet** table, not the sampled cohort, or
occupancy counts would be understated and the redundancy margin wrong.

In [ ]:
fleet = (pl.scan_parquet(FLEET_GLOB)
           .with_columns(pl.col("date").cast(pl.Date))
           .filter(pl.col("pod_slot_num").is_not_null()))

# Daily per-tome state over the whole fleet: size, and how many members failed that day.
tome_daily = (fleet.group_by(["date", "vault_id", "pod_slot_num"])
                   .agg(tome_size       = pl.len(),
                        tome_failures   = pl.col("failure").sum(),
                        tome_cap_tb     = (pl.col("capacity_bytes").median() / 1e12).round(1))
                   .collect())

tome_daily = tome_daily.with_columns(
    # Parity budget rises with capacity, following Backblaze's published practice of offsetting
    # longer rebuild times on bigger drives. An explicit modelling assumption of the simulator.
    parity = pl.when(pl.col("tome_cap_tb") >= 14).then(pl.lit(5, dtype=pl.Int8))   # 15+5
              .when(pl.col("tome_cap_tb") >= 10).then(pl.lit(4, dtype=pl.Int8))    # 16+4
              .otherwise(pl.lit(3, dtype=pl.Int8))                                 # 17+3
).sort(
    # cum_sum below walks rows in frame order, so the sort is load-bearing, not cosmetic:
    # an unsorted frame produces a cumulative count over shuffled dates.
    ["vault_id", "pod_slot_num", "date"]
).with_columns(
    tome_cum_failures = pl.col("tome_failures").cum_sum()
                          .over(["vault_id", "pod_slot_num"])
).with_columns(
    redundancy_margin = (pl.col("parity") - pl.col("tome_cum_failures")).cast(pl.Int16)
)

print(f"tome-day rows: {len(tome_daily):,}")
print(tome_daily.head(5).to_pandas().to_string(index=False))
print("\n=== redundancy margin distribution (lower = closer to data loss) ===")
print(tome_daily["redundancy_margin"].value_counts()
        .sort("redundancy_margin").head(10).to_pandas().to_string(index=False))

In [ ]:
lf = lf.join(tome_daily.lazy().select(
                ["date", "vault_id", "pod_slot_num", "tome_size",
                 "tome_failures", "parity", "tome_cum_failures", "redundancy_margin"]),
             on=["date", "vault_id", "pod_slot_num"], how="left")

lf = lf.with_columns(
    # excludes this drive itself, so it is a genuine neighbour signal
    tome_siblings_failed = (pl.col("tome_cum_failures") - pl.col("failure")).clip(0, None),
    tome_occupancy_ratio = (pl.col("tome_size") / 20.0),
)
print("tome-context features joined")

## 8. Temporal split

Three guards, all necessary:

- **Warm-up excluded from modelling.** December 2025 exists only so trailing windows are
  defined for early January. It carries no labels and is dropped here.
- **Embargo gap.** A training row dated Feb 15 carries a label describing events up to Feb 22.
  Without a gap, that label overlaps the evaluation period and the split leaks. The embargo is
  at least as long as the longest horizon and is discarded entirely.
- **No drive spans train and test.** Even with the time split, the same physical drive
  appearing on both sides lets the model memorise drive identity instead of learning
  degradation.

In [ ]:
def split_of(col: pl.Expr) -> pl.Expr:
    return (pl.when(col < Q1_START).then(pl.lit("warmup"))
              .when((col >= TRAIN_START) & (col <= TRAIN_END)).then(pl.lit("train"))
              .when((col >= EMBARGO_START) & (col <= EMBARGO_END)).then(pl.lit("embargo"))
              .when((col >= CALIB_START) & (col <= CALIB_END)).then(pl.lit("calib"))
              .when((col >= TEST_START) & (col <= TEST_END)).then(pl.lit("test"))
              .otherwise(pl.lit("other")).alias("split"))

lf = lf.with_columns(split_of(pl.col("date")))

print("=== rows per split (before filtering) ===")
print(lf.group_by("split").agg(
        rows = pl.len(),
        pos  = pl.col(f"fail_in_{PRIMARY_HORIZON}d").sum(),
      ).collect().sort("split").to_pandas().to_string(index=False))

In [ ]:
feat = (lf.filter(pl.col("split").is_in(["train", "calib", "test"]))
          .filter(pl.col(f"fail_in_{PRIMARY_HORIZON}d").is_not_null())
          .collect())

print(f"modelling rows: {len(feat):,}  |  columns: {len(feat.columns):,}")

# Enforce drive disjointness: a drive appearing in both train and test is removed from train,
# so the evaluation stays honest.
tr_ser = set(feat.filter(pl.col("split") == "train")["serial_number"].unique().to_list())
te_ser = set(feat.filter(pl.col("split") == "test")["serial_number"].unique().to_list())
overlap = tr_ser & te_ser
print(f"\ndrives in both train and test: {len(overlap):,}")

if overlap:
    before = len(feat)
    feat = feat.filter(~((pl.col("split") == "train") &
                          pl.col("serial_number").is_in(list(overlap))))
    print(f"dropped {before - len(feat):,} train rows to enforce drive disjointness")

print("\n=== final split summary ===")
print(feat.group_by("split").agg(
        rows   = pl.len(),
        drives = pl.col("serial_number").n_unique(),
        pos    = pl.col(f"fail_in_{PRIMARY_HORIZON}d").sum(),
      ).sort("split").to_pandas().to_string(index=False))

## 9. Leakage and sanity tests

These are the guardrails for the failure modes that silently ruined the previous attempt.
They are written as assertions so a regression stops the notebook instead of producing a
plausible-looking but worthless feature table.

**Test 1 - deltas actually vary.** The old file's three delta columns were constant zero
across every row. Any near-zero-variance delta column means the time series is broken upstream.

**Test 2 - causality.** Recompute a feature using only rows up to date *t* and confirm it
matches the value computed from the full history. If truncating the future changes a
feature, that feature was reading the future.

**Test 3 - horizon labels are nested.** A 7-day positive must also be a 14-day positive.
This catches the identical-label bug directly.

In [ ]:
delta_cols = [c for c in feat.columns if "_delta_" in c]
stats = feat.select([pl.col(c).std().alias(c) for c in delta_cols]).to_pandas().T[0]

dead = stats[(stats.isna()) | (stats < 1e-12)]

# A vendor-conditional attribute can legitimately be flat on a sample dominated by vendors
# that do not report it, so those are reported separately rather than counted as failures.
# The universally-reported attributes have no such excuse and are the real test.
vendor_conditional = tuple(c.replace("_raw", "") for c in SPARSE_COLS)
core_dead   = [c for c in dead.index if not c.startswith(vendor_conditional)]
vendor_dead = [c for c in dead.index if c.startswith(vendor_conditional)]

print(f"delta features checked        : {len(delta_cols)}")
print(f"zero-variance, core           : {len(core_dead)}")
print(f"zero-variance, vendor-cond.   : {len(vendor_dead)}  (expected, not a failure)")
if core_dead:
    print("\nDEAD CORE COLUMNS:")
    print(stats[core_dead].to_string())

assert not core_dead, (
    f"{len(core_dead)} universally-reported delta features are constant - the panel is not a "
    "proper daily time series (this is exactly how the earlier processed file failed)")
print("\nTEST 1 PASS - every core delta feature carries variance")

print("\nnon-zero rate for the headline deltas:")
for c in ["smart_5_raw_delta_7d", "smart_197_raw_delta_7d", "smart_187_raw_delta_7d"]:
    if c in feat.columns:
        nz = (feat[c].fill_null(0) != 0).mean()
        print(f"  {c:34s} {100*nz:6.3f}% non-zero   std={stats.get(c, float('nan')):.4f}")

In [ ]:
# TEST 2 - causality by truncation.
CUT = date(2026, 2, 20)
probe_col = "smart_5_raw_delta_7d"

probe_serials = (feat.filter(pl.col("split") == "test")["serial_number"]
                     .unique().head(300).to_list())

def recompute(upto: date | None):
    q = (pl.scan_parquet(PANEL_GLOB)
           .with_columns(pl.col("date").cast(pl.Date))
           .filter(pl.col("serial_number").is_in(probe_serials)))
    if upto is not None:
        q = q.filter(pl.col("date") <= upto)
    q = q.sort(["serial_number", "date"])
    base = pl.col("smart_5_raw").forward_fill().over("serial_number")
    return (q.with_columns((base - base.shift(7).over("serial_number")).alias(probe_col))
             .filter(pl.col("date") == CUT)
             .select(["serial_number", probe_col])
             .sort("serial_number").collect())

full_hist  = recompute(None)
trunc_hist = recompute(CUT)

merged = full_hist.join(trunc_hist, on="serial_number", how="inner",
                        suffix="_trunc")
diff = (merged[probe_col].fill_null(-999) -
        merged[f"{probe_col}_trunc"].fill_null(-999)).abs().max()

print(f"probed {len(merged):,} drives on {CUT}")
print(f"max |full-history - truncated| = {diff}")
assert diff == 0, f"{probe_col} changes when future rows are removed -> it leaks the future"
print("\nTEST 2 PASS - feature is causal (truncating the future changes nothing)")

In [ ]:
# TEST 3 - horizon labels must be nested and distinct.
h7  = feat["fail_in_7d"].fill_null(0)
h14 = feat["fail_in_14d"].fill_null(0)
h30 = feat["fail_in_30d"].fill_null(0)

viol_7_14  = ((h7 == 1) & (h14 == 0)).sum()
viol_14_30 = ((h14 == 1) & (h30 == 0)).sum()

print(f"positives: 7d={h7.sum():,}  14d={h14.sum():,}  30d={h30.sum():,}")
print(f"violations 7d positive but not 14d : {viol_7_14:,}")
print(f"violations 14d positive but not 30d: {viol_14_30:,}")

assert viol_7_14 == 0 and viol_14_30 == 0, "horizon labels are not nested"
assert h7.sum() < h14.sum() < h30.sum(), \
    "horizon positive counts are not strictly increasing - labels may be duplicated"
print("\nTEST 3 PASS - labels nested, distinct and strictly increasing")

In [ ]:
# Additional hygiene: the failure day itself.
# On the failure day SMART often already reflects the crash, so those rows are trivially
# separable and would inflate offline scores while teaching nothing about EARLY warning.
# We keep them but flag them, so notebook 04 can report metrics with and without.
feat = feat.with_columns(
    is_failure_day = (pl.col("time_to_failure") == 0).fill_null(False).cast(pl.Int8))

print(f"failure-day rows flagged: {feat['is_failure_day'].sum():,}")

print("\n=== null rate of the top engineered features ===")
eng = [c for c in feat.columns if any(k in c for k in
       ["_delta_", "_zdrift_", "_max_", "_std_", "days_since", "redundancy", "margin"])]
nr = feat.select([pl.col(c).is_null().mean().alias(c) for c in eng]).to_pandas().T[0]
print(nr.sort_values(ascending=False).head(15).mul(100).round(2).to_string())

## 10. Save the feature table

In [ ]:
DROP = ["failure_date", "last_seen", "any_anomaly", "day_index", "first_anomaly_idx"]
out = feat.drop([c for c in DROP if c in feat.columns])

out.write_parquet(FEATURES, compression="zstd")
size_mb = os.path.getsize(FEATURES) / 1e6

print(f"saved {FEATURES}")
print(f"  rows    : {len(out):,}")
print(f"  columns : {len(out.columns):,}")
print(f"  size    : {size_mb:.1f} MB")

feature_cols = [c for c in out.columns if c not in
                {"date", "serial_number", "model", "manufacturer", "split", "sample_weight",
                 "failure", "censored", "time_to_failure", "days_to_exit", "is_failure_day",
                 "datacenter", "cluster_id", "vault_id", "pod_id", "pod_slot_num"}
                and not c.startswith("fail_in_")]
print(f"\nmodel-ready feature columns: {len(feature_cols):,}")
pd.Series(feature_cols).to_csv(f"{DRIVE_ROOT}/feature_columns.csv", index=False, header=["feature"])

In [ ]:
# Quick signal check: do the engineered features actually separate the classes?
# Not a model, just a sanity read before investing in training.
y = out[f"fail_in_{PRIMARY_HORIZON}d"].to_numpy()
num_cols = [c for c in feature_cols if out[c].dtype in
            (pl.Float32, pl.Float64, pl.Int8, pl.Int16, pl.Int32, pl.Int64)]

rows = []
for c in num_cols:
    v = out[c].to_numpy().astype("float64")
    m = ~np.isnan(v)
    if m.sum() < 1000 or len(np.unique(v[m])) < 2:
        continue
    pos, neg = v[m & (y == 1)], v[m & (y == 0)]
    if len(pos) < 30 or len(neg) < 30:
        continue
    sd = np.sqrt((pos.var() + neg.var()) / 2)
    if sd > 0:
        rows.append((c, abs(pos.mean() - neg.mean()) / sd))

sep = pd.DataFrame(rows, columns=["feature", "cohens_d"]).sort_values("cohens_d", ascending=False)
print("=== top 25 features by standardised mean difference ===")
print(sep.head(25).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
top = sep.head(20).iloc[::-1]
ax.barh(top.feature, top.cohens_d, color="#1f7a6f")
ax.set_xlabel("|Cohen's d|"); ax.set_title(f"Univariate separation, {PRIMARY_HORIZON}-day horizon")
plt.tight_layout(); plt.show()

## Next steps

`features.parquet` now holds a leakage-tested modelling table with an embargoed temporal
split, censoring handled, and tome-context features from real topology.

**04 - Prediction.** Start with the two baselines that must be beaten before any gradient
boosting result means anything: the industry-standard "any non-zero `smart_5/187/197/198`"
threshold rule, and logistic regression. Then LightGBM with `GroupKFold` by serial nested in a
time-series split, a GRU on 30-day windows, and a discrete-time hazard model that uses the
censored drives instead of discarding them. Calibrate on the `calib` split; report PR-AUC,
precision at a fixed false-alarm budget, Brier and ECE with bootstrap CIs.

**05 - Healing policy.** Action space, expected-cost threshold derived from failure versus
false-alarm costs, and the hard tome-parity and concurrency rules that override the model.

**06 - Reliability checker.** The counterfactual replay against the tome-state fleet
simulator, producing the five-axis scorecard: correctness, safety, timeliness, efficiency and
stability.

One caveat to carry forward: the `univariate separation` ranking above is computed on the
modelling table including the `test` split. Use it only as a smoke test, never for feature
selection - selecting features on the test split is itself a leak.